In [1]:
import sys
from demosat_dictionary_interface.command import CommandDictionary
from demosat_dictionary_interface.channel import ChannelDictionary

In [2]:
# TO DO: Move this resource business into tts_dictionary_interface
# modern importlib.resources is preferred for Python 3.9+
# Python 3.7 and 3.8 have a more limited version of importlib.resources
if sys.version_info >= (3, 9):
    import importlib.resources as resources
    HAS_MODERN_RESOURCES = True
else:
    # Fallback for Python 3.6.8 and older environments
    import pkg_resources
    HAS_MODERN_RESOURCES = False

def get_resource_path(package, resource):
    """
    Unified helper to find file paths across different Python versions.
    """
    if HAS_MODERN_RESOURCES:
        # Modern API: returns a path object using the files() traversal
        return str(resources.files(package).joinpath(resource))
    else:
        # Legacy fallback for Python 3.6: uses pkg_resources
        return pkg_resources.resource_filename(package, resource)
try:
    package_name = "demosat_dict.dictionaries.v1"
    command_xml = get_resource_path(package_name, "command.xml")
    channel_xml = get_resource_path(package_name, "Channel.xml")
except (ImportError, FileNotFoundError):
    print("Error: demosat_dict package or dictionary files not found.")


# DemoSat Dictionary Interface

This library is meant to be a common interface between all projects that use TTS. It consumes AMPCS dicitonaries (or ostensibly others, but right now all we have is AMPCS) into a pyhtonic object, which makes it easier to explore and provides a common interface so projects can share common implementations of tools downstream of dictionarys like sequencing and simulation by just swapping out the dictiony class extension

We begin by reading in the dictionaries

In [6]:
cmd_dict = CommandDictionary(command_xml)
eha_dict = ChannelDictionary(channel_xml)
print(f'Parsed {len(cmd_dict)} commands')
print(f'Parsed {len(eha_dict)} channels (note, this is a bug in the len() method for channels)')

Parsed 25 commands
Parsed 24 channels (note, this is a bug in the len() method for channels)


In [7]:
sci_cmd = cmd_dict['DO_SCIENCE']
print(f'Stem: {sci_cmd.stem}')
print(f'Opcode: {sci_cmd.opcode}')
print(f'Found {len(sci_cmd.args)} argument(s) in {sci_cmd.stem}')
for arg in sci_cmd.args:
    print(f'\tname: {arg.name} type: {arg.type}')

yaw_cmd = cmd_dict['ADCS_YAW']
print(f'Stem: {yaw_cmd.stem}')
print(f'Opcode: {yaw_cmd.opcode}')
print(f'Found {len(yaw_cmd.args)} argument(s) in {yaw_cmd.stem}')
for arg in yaw_cmd.args:
    print(f'\tname: {arg.name} type: {arg.type}')

Stem: DO_SCIENCE
Opcode: 0x0013
Found 2 argument(s) in DO_SCIENCE
	name: hemisphere type: ENUM.HEMISPHERE
	name: day_night type: ENUM.DAY_NIGHT
Stem: ADCS_YAW
Opcode: 0x0020
Found 1 argument(s) in ADCS_YAW
	name: roll_angle type: F32


In [8]:
batt_chan = eha_dict['BATTERY_VOLTAGE']
print("Channel: {} ({})".format(batt_chan.name, batt_chan.id))
print("  Chanel Type: {}".format(batt_chan.type))
print("  AMPCS Data Type: {}".format(batt_chan.ampcs_type))
print("  Description: {}".format(batt_chan.description))

batt_chan = eha_dict['HEATER_1_ENABLED']
print("Channel: {} ({})".format(batt_chan.name, batt_chan.id))
print("  Chanel Type: {}".format(batt_chan.type))
print("  AMPCS Data Type: {}".format(batt_chan.ampcs_type))
print("  Description: {}".format(batt_chan.description))


Channel: BATTERY_VOLTAGE (EPS-0001)
  Chanel Type: float
  AMPCS Data Type: eu
  Description: Main Battery Voltage (Raw DN to Volts)
Channel: HEATER_1_ENABLED (EPS-0011)
  Chanel Type: enum
  AMPCS Data Type: state
  Description: State of Heater 1 (Extracted from Status Word)


In [15]:
for a in cmd_dict['DO_SCIENCE'].args: print(a)

24